In [6]:
!pip install -q openai-whisper
!pip install -q transformers accelerate
!pip install -q fastapi uvicorn pyngrok
!pip install -q pillow python-multipart peft



In [ ]:
 !ngrok config add-authtoken [Token]

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
# %%writefile server.py

# import torch
# import shutil
# import uvicorn
# import whisper

# from PIL import Image
# from fastapi import FastAPI, UploadFile, File
# from pydantic import BaseModel
# from pyngrok import ngrok

# from transformers import (
#     BlipProcessor,
#     BlipForConditionalGeneration,
#     AutoTokenizer,
#     AutoModelForCausalLM
# )

# from peft import PeftModel


# # ==============================
# # DEVICE
# # ==============================

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)


# # ==============================
# # LOAD WHISPER
# # ==============================

# print("Loading Whisper model...")

# whisper_model = whisper.load_model("base")

# print("Whisper loaded")


# # ==============================
# # LOAD BLIP IMAGE CAPTION
# # ==============================

# print("Loading BLIP caption model...")

# caption_processor = BlipProcessor.from_pretrained(
#     "Salesforce/blip-image-captioning-base"
# )

# caption_model = BlipForConditionalGeneration.from_pretrained(
#     "Salesforce/blip-image-captioning-base"
# ).to(device)

# print("BLIP loaded")


# # ==============================
# # LOAD QWEN BASE MODEL
# # ==============================

# print("Loading Qwen base model...")

# model_id = "Qwen/Qwen2.5-3B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(model_id)

# base_model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# print("Base model loaded")


# # ==============================
# # LOAD QLORA CHECKPOINT
# # ==============================

# print("Loading LoRA checkpoint...")

# # TODO: BẠN THAY ĐƯỜNG DẪN CHECKPOINT Ở ĐÂY
# LORA_PATH = "/content/drive/MyDrive/video-summary-finetune/runs/20260311_040836_qwen_news_qlora/archive/checkpoint-650"

# summary_model = PeftModel.from_pretrained(
#     base_model,
#     LORA_PATH
# )

# # optional: merge LoRA → full model (deploy nhanh hơn)
# try:
#     summary_model = summary_model.merge_and_unload()
#     print("LoRA merged into base model")
# except:
#     print("Running with LoRA adapter")

# summary_model.eval()

# print("Qwen + LoRA loaded successfully")


# # ==============================
# # FASTAPI APP
# # ==============================

# app = FastAPI()


# # ==============================
# # SPEECH TO TEXT
# # ==============================

# @app.post("/transcribe")
# async def transcribe(file: UploadFile = File(...)):

#     temp_file = "temp_audio.wav"

#     with open(temp_file, "wb") as buffer:
#         shutil.copyfileobj(file.file, buffer)

#     result = whisper_model.transcribe(temp_file)

#     return result


# # ==============================
# # IMAGE CAPTION
# # ==============================

# @app.post("/caption")
# async def caption(file: UploadFile = File(...)):

#     temp_file = "temp_image.jpg"

#     with open(temp_file, "wb") as buffer:
#         shutil.copyfileobj(file.file, buffer)

#     image = Image.open(temp_file).convert("RGB")

#     inputs = caption_processor(
#         images=image,
#         return_tensors="pt"
#     ).to(device)

#     out = caption_model.generate(**inputs)

#     caption = caption_processor.decode(
#         out[0],
#         skip_special_tokens=True
#     )

#     return {"caption": caption}


# # ==============================
# # TEXT GENERATION / SUMMARY
# # ==============================

# class TextRequest(BaseModel):
#     text: str


# @app.post("/generate")
# async def generate(req: TextRequest):

#     inputs = tokenizer(
#         req.text,
#         return_tensors="pt"
#     ).to(summary_model.device)

#     outputs = summary_model.generate(
#         **inputs,
#         max_new_tokens=300,
#         temperature=0.7
#     )

#     result = tokenizer.decode(
#         outputs[0],
#         skip_special_tokens=True
#     )

#     return {"result": result}


# # ==============================
# # START NGROK
# # ==============================

# public_url = ngrok.connect(8000)

# print("PUBLIC API:", public_url)


# # ==============================
# # RUN SERVER
# # ==============================

# if __name__ == "__main__":
#     uvicorn.run(app, host="0.0.0.0", port=8000)


Overwriting server.py


In [3]:
%%writefile server.py
import torch
import whisper
from PIL import Image
import shutil

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM
)

from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel


from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import whisper
import shutil
import torch
import whisper
from PIL import Image
import shutil

print("Loading Whisper model...")
whisper_model = whisper.load_model("base")

print("Loading BLIP caption model...")
caption_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to("cuda")

print("Loading Qwen summary model...")
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

summary_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

summary_model.eval()

print("All models loaded successfully")


app = FastAPI()

# load whisper model
whisper_model = whisper.load_model("base")


@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):

    temp_file = "temp_audio.wav"

    with open(temp_file, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    result = whisper_model.transcribe(temp_file)

    return result

@app.post("/caption")
async def caption(file: UploadFile = File(...)):

    temp_file = "temp_image.jpg"

    with open(temp_file, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    image = Image.open(temp_file).convert("RGB")

    inputs = caption_processor(
        images=image,
        return_tensors="pt"
    ).to("cuda")

    out = caption_model.generate(**inputs)

    caption = caption_processor.decode(
        out[0],
        skip_special_tokens=True
    )

    return {"caption": caption}


class TextRequest(BaseModel):
    text: str


@app.post("/generate")
async def generate(req: TextRequest):

    inputs = tokenizer(
        req.text,
        return_tensors="pt"
    ).to(summary_model.device)

    outputs = summary_model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {"result": result}

# start ngrok
public_url = ngrok.connect(8000)
print("PUBLIC API:", public_url)


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


Writing server.py


In [ ]:
!uvicorn server:app --host 0.0.0.0 --port 8000

Using device: cuda
Loading Whisper model...
Whisper loaded
Loading BLIP caption model...
The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 473/473 [00:00<00:00, 721.50it/s, Materializing param=vision_model.post_layernorm.weight]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are presen